<img src="https://raw.githubusercontent.com/Delarbol/semillero/master/Images-DS/alt_banner.png" width=100%>
<h1 style="margin:0">Inteligencia Artificial Aplicada a la Economía II</h1>
<h2 style="margin:.35em 0 0 0">¿Qué es un LLM? De redes neuronales y transformers a una llamada desde Python</h2>
<p style="margin:.8em 0 0 0"><strong>Lunes 21 y viernes 25 de septiembre de 2026 · 7:00–9:00 a. m.</strong></p>
</div>

> **Idea guía:** un modelo grande de lenguaje no “busca una respuesta guardada”. Recibe tokens, calcula probabilidades y genera una continuación. Su capacidad aparece al repetir esta operación con una red neuronal de gran escala y un contexto amplio.

# Propósito de la sesión

Al terminar, usted debería poder explicar —sin recurrir a la frase “es una caja negra”—:

- qué significan **LLM** (*Large Language Model* o modelo grande de lenguaje), **token**, **parámetro**, **embedding**, **atención** y **transformer**;
- cómo se conecta la predicción del siguiente token con una red neuronal;
- qué hace la atención y por qué un transformer puede usar el contexto;
- cómo se genera texto token por token;
- qué capacidades y límites se derivan de este mecanismo;
- cómo enviar una solicitud a un LLM desde Python mediante una **API** (*Application Programming Interface* o interfaz de programación de aplicaciones);
- cómo proteger una clave, controlar costos y conservar trazabilidad.

No se espera que usted programe un transformer completo. Sí se espera que pueda seguir los cálculos pequeños, interpretar resultados y adaptar la llamada final.

## Ruta de 120 minutos

| Bloque | Tiempo | Pregunta central |
|---|---:|---|
| 1. Modelo de lenguaje y tokens | 25 min | ¿Qué predice realmente un LLM? |
| 2. Red neuronal y embeddings | 25 min | ¿Dónde se aprende el patrón? |
| 3. Atención y transformer | 30 min | ¿Cómo usa el contexto? |
| 4. Generación, entrenamiento y límites | 20 min | ¿Por qué puede responder bien y también alucinar? |
| 5. Llamada desde Python | 20 min | ¿Cómo se integra un LLM en un programa? |

Trabaje las preguntas breves con su pareja. Antes de abrir una respuesta desplegable, escriban una predicción.

# 0. Preparación

Los ejemplos conceptuales usan `numpy`, `pandas`, `matplotlib` y `seaborn`. La llamada real del final usa el SDK (*Software Development Kit*, kit de desarrollo de software) oficial de OpenAI, pero está **desactivada por defecto**.

Si una librería falta, instálela en una celda separada y reinicie el kernel:

```python
%pip install -q numpy pandas matplotlib seaborn openai
```

La llamada real puede generar consumo facturable. No la active hasta que el docente lo indique.

In [ ]:
import os
import re
from collections import Counter, defaultdict
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.set_printoptions(precision=3, suppress=True)
pd.set_option('display.max_colwidth', 120)
sns.set_theme(style='whitegrid', context='notebook')

SEMILLA = 42
rng = np.random.default_rng(SEMILLA)
print('Entorno listo. Semilla reproducible:', SEMILLA)

# 1. ¿Qué es un LLM?

**LLM** significa *Large Language Model*: **modelo grande de lenguaje**.

La expresión contiene tres ideas distintas:

| Palabra | Significado operativo |
|---|---|
| **Modelo** | Una función matemática con parámetros aprendidos a partir de datos. |
| **De lenguaje** | Asigna probabilidades a secuencias de tokens y, al generar, estima cuál token podría continuar el contexto. |
| **Grande** | Tiene muchos parámetros, fue entrenado con grandes volúmenes de datos y requirió bastante cómputo. “Grande” no significa infalible. |

Una forma compacta de escribir su tarea es:

$$
P(t_{n+1}\mid t_1,t_2,\ldots,t_n)
$$

Se lee: “probabilidad del próximo token, dado todo el contexto anterior”.

## Tres distinciones importantes

Un LLM:

- **no es una base de datos:** sus parámetros no son filas que podamos consultar como una tabla;
- **no es necesariamente un buscador:** sin una herramienta externa, puede producir texto sin consultar fuentes actuales;
- **no es una calculadora de verdad:** optimiza una continuación plausible, no una garantía de veracidad.

Esto no significa que “solo repita”. Aprende regularidades lingüísticas y conceptuales suficientemente complejas para resumir, clasificar, traducir, explicar y ayudar a programar. El punto es comprender de dónde salen esas capacidades y qué no garantizan.

### Comprobación rápida 1

Clasifique cada frase como verdadera o falsa y justifique en una oración.

1. Si un LLM asigna alta probabilidad a una continuación, entonces esa continuación es verdadera.
2. Un modelo de lenguaje puede generar una oración al elegir repetidamente el siguiente token.
3. “Grande” significa que el modelo siempre conoce información reciente.

<details>
<summary>Ver retroalimentación después de responder</summary>

1. **Falsa.** Plausibilidad lingüística y verdad no son equivalentes.
2. **Verdadera.** La generación autoregresiva repite predicción y selección de tokens.
3. **Falsa.** La actualidad depende de los datos, el acceso a herramientas y el contexto disponible.
</details>

# 2. Del texto a los tokens

Una computadora no recibe directamente “ideas”: recibe números. Antes de entrar a la red, el texto se divide en unidades llamadas **tokens**.

Un token puede ser:

- una palabra completa;
- una parte de una palabra;
- un signo de puntuación;
- un espacio u otra unidad especial, según el tokenizador.

Los tokenizadores reales suelen usar **subpalabras**. Así pueden representar palabras poco frecuentes sin reservar una entrada independiente para cada palabra posible.

> El número de tokens no tiene por qué coincidir con el número de palabras.

## Ejemplo guiado · tokenizador didáctico

El siguiente tokenizador separa palabras y signos. **No reproduce** el tokenizador de un modelo comercial; solo hace visible la transformación.

In [ ]:
def tokenizar_simple(texto):
    """Tokenización didáctica: palabras o signos de puntuación."""
    return re.findall(r"\w+|[^\w\s]", texto.lower(), flags=re.UNICODE)


frase = 'La inflación anual bajó a 5,2 %, según el informe.'
tokens = tokenizar_simple(frase)

print('Texto original:', frase)
print('Tokens:', tokens)
print('Número de palabras aproximado:', len(frase.split()))
print('Número de tokens didácticos:', len(tokens))

## Vocabulario e identificadores

El **vocabulario** es el conjunto de tokens que el modelo puede representar. A cada token se le asigna un identificador entero. El identificador no expresa por sí mismo significado: es una dirección para buscar un vector.

In [ ]:
vocabulario_demo = sorted(set(tokens + ['empleo', 'desempleo', '<inicio>', '<fin>']))
token_a_id = {token: i for i, token in enumerate(vocabulario_demo)}

tabla_vocabulario = pd.DataFrame({
    'token': vocabulario_demo,
    'id': [token_a_id[t] for t in vocabulario_demo],
})
tabla_vocabulario.head(12)

### Su turno 1 · adaptar el ejemplo

1. Cambie `texto_estudiante` por una oración económica que incluya un número y puntuación.
2. Ejecute el tokenizador.
3. Responda debajo:
   - ¿cuántas palabras contó usted visualmente?;
   - ¿cuántos tokens produjo la función?;
   - ¿qué unidades explican la diferencia?

In [ ]:
texto_estudiante = 'El desempleo fue 9,8 % en agosto.'  # Reemplace por su ejemplo
tokens_estudiante = tokenizar_simple(texto_estudiante)

print(tokens_estudiante)
print('Cantidad:', len(tokens_estudiante))

respuesta_tokens = 'Escriba aquí su interpretación.'
print(respuesta_tokens)

# 3. Un modelo de lenguaje mínimo: contar continuaciones

Antes de usar redes neuronales, construiremos un modelo muy pequeño basado en conteos. Será un **modelo bigrama**: solo mira el token inmediatamente anterior.

Si en el corpus aparece muchas veces “inflación sube” y pocas veces “inflación baja”, el modelo asignará mayor probabilidad a `sube` después de `inflación`.

$$
P(\text{siguiente}=b\mid\text{actual}=a)
=\frac{\text{conteo}(a,b)}{\sum_j \text{conteo}(a,j)}
$$

Este modelo es limitado, pero deja visible la misma pregunta que luego resolverá una red mucho mayor: **¿qué viene después dado un contexto?**

In [ ]:
corpus = [
    'la inflación sube cuando aumentan los costos',
    'la inflación baja cuando cae la demanda',
    'la inflación sube y reduce el poder adquisitivo',
    'el empleo crece cuando aumenta la actividad',
    'el empleo cae cuando baja la actividad',
    'la tasa sube para contener la inflación',
    'la tasa baja para estimular la actividad',
]

conteos = defaultdict(Counter)

for oracion in corpus:
    secuencia = ['<inicio>'] + tokenizar_simple(oracion) + ['<fin>']
    for actual, siguiente in zip(secuencia[:-1], secuencia[1:]):
        conteos[actual][siguiente] += 1

print('Continuaciones observadas después de “inflación”:', conteos['inflación'])

In [ ]:
def probabilidades_conteo(token_actual):
    frecuencias = conteos[token_actual]
    total = sum(frecuencias.values())
    if total == 0:
        return pd.DataFrame(columns=['siguiente', 'conteo', 'probabilidad'])
    return (
        pd.DataFrame({
            'siguiente': list(frecuencias.keys()),
            'conteo': list(frecuencias.values()),
        })
        .assign(probabilidad=lambda df: df['conteo'] / total)
        .sort_values('probabilidad', ascending=False)
        .reset_index(drop=True)
    )


probabilidades_conteo('inflación')

## Generar también implica seleccionar

Con las probabilidades calculadas hay varias estrategias:

- elegir siempre el token de mayor probabilidad;
- muestrear aleatoriamente según la distribución;
- modificar la distribución antes de muestrear.

Elegir siempre el máximo hace la salida más estable, pero puede volverla repetitiva. Muestrear introduce diversidad, pero también riesgo.

In [ ]:
def siguiente_por_conteo(token_actual, rng_local):
    tabla = probabilidades_conteo(token_actual)
    if tabla.empty:
        return '<fin>'
    return rng_local.choice(
        tabla['siguiente'].to_numpy(),
        p=tabla['probabilidad'].to_numpy(),
    )


for i in range(5):
    print(i + 1, 'inflación →', siguiente_por_conteo('inflación', rng))

### Su turno 2 · evidencia y límites

Copie y adapte las llamadas anteriores para estudiar `empleo`, `tasa` y una palabra ausente del corpus.

Responda:

1. ¿Qué hace el modelo cuando no ha visto el token?
2. ¿Por qué este bigrama no distingue entre “la tasa baja” y “la inflación baja” al predecir después de `baja`?
3. ¿Qué información adicional necesitaría un modelo menos limitado?

In [ ]:
tokens_a_probar = ['empleo', 'tasa', 'productividad']

for token in tokens_a_probar:
    print(f'\nDespués de {token!r}:')
    display(probabilidades_conteo(token))

respuesta_bigrama = 'Escriba aquí sus respuestas 1–3.'
print(respuesta_bigrama)

# 4. La conexión con redes neuronales

Una **red neuronal** es una función compuesta por operaciones numéricas sencillas. Sus parámetros —matrices y vectores de números— se ajustan durante el entrenamiento para reducir un error.

En un modelo de lenguaje simplificado:

1. un token se convierte en un vector (**embedding**);
2. varias capas transforman ese vector usando el contexto;
3. una capa de salida produce un puntaje para cada token del vocabulario;
4. `softmax` convierte los puntajes en probabilidades;
5. la pérdida mide cuánto se alejó la predicción del token correcto;
6. el descenso de gradiente ajusta los parámetros.

Los puntajes anteriores a `softmax` se llaman **logits**. Pueden ser positivos o negativos y todavía no suman uno.

Los **logits** son los valores numéricos “en bruto” que produce un modelo **antes de convertirlos en probabilidades**.

Por ejemplo, imagina que un modelo clasifica una imagen entre tres animales y produce:

$$
\text{logits} = [2.3,\; 0.8,\; -1.2]
$$

correspondientes a:

$$
[\text{gato},\text{perro},\text{caballo}]
$$

Esos números **no son probabilidades**: no tienen que estar entre 0 y 1 ni sumar 1. Solo expresan qué opciones está favoreciendo el modelo. Aquí, `gato` tiene el logit más alto.

Luego normalmente aplicamos **softmax**:

$$
P(y_i)=\frac{e^{z_i}}{\sum_j e^{z_j}}
$$

donde \(z_i\) es cada logit. En este ejemplo, aproximadamente obtendríamos:

$$
[0.80,\;0.18,\;0.02]
$$

Ahora sí tenemos probabilidades que suman 1.

En un **LLM como GPT**, ocurre algo parecido pero a una escala enorme. Supón que después del texto:

> “La capital de Colombia es”

el modelo tiene que decidir cuál token viene después. Puede generar logits como:

```text
" Bogotá"     12.4
" Medellín"    7.1
" Cali"        5.8
" París"      -1.2
...
```

Si el vocabulario tiene, por ejemplo, 100.000 tokens, el modelo genera **un logit para cada uno de esos 100.000 tokens**. Después se transforman en una distribución de probabilidades:

```text
" Bogotá"    → 0.94
" Medellín"  → 0.02
" Cali"      → 0.01
...
```

Y de esa distribución se selecciona el siguiente token.

Un detalle importante: **la diferencia entre los logits importa más que sus valores absolutos**. Por ejemplo:

$$
[10,8,2]
$$

y

$$
[5,3,-3]
$$

producen exactamente la misma distribución softmax, porque las diferencias entre los valores son iguales.

En una red neuronal, los logits normalmente aparecen justo en la **última capa lineal**:

$$
h \rightarrow Wh+b \rightarrow \text{logits} \rightarrow \text{softmax}
$$

En un Transformer/LLM, \(h\) es la representación interna que construyó el Transformer para el token actual. La última proyección la transforma en un vector cuyo tamaño es el vocabulario:

$$
\mathbf{z}=W\mathbf{h}+b
$$

y ese vector \(\mathbf{z}\) son los **logits**.

Una forma sencilla de recordarlo es:

> **Logits = qué tanto favorece el modelo cada opción.**
> **Softmax = convierte esas preferencias en probabilidades.**

Y esto conecta directamente con algo muy interesante de los LLM: la **temperature** actúa precisamente sobre los logits antes del softmax:

$$
P_i =
\frac{e^{z_i/T}}
{\sum_j e^{z_j/T}}
$$

Con temperatura baja, las diferencias entre logits pesan más y el modelo se vuelve más determinista. Con temperatura alta, las probabilidades se distribuyen más y las respuestas se vuelven más variadas.


## De logits a probabilidades: `softmax`

Para un vector de logits $z$, la probabilidad del token $i$ es:

$$
\operatorname{softmax}(z_i)=\frac{e^{z_i}}{\sum_j e^{z_j}}
$$

Observe que aumentar un logit eleva su probabilidad y reduce, indirectamente, la participación de los demás.

In [ ]:
def softmax(valores, axis=-1):
    valores = np.asarray(valores, dtype=float)
    desplazados = valores - np.max(valores, axis=axis, keepdims=True)
    exponentes = np.exp(desplazados)
    return exponentes / exponentes.sum(axis=axis, keepdims=True)


candidatos = ['sube', 'baja', 'crece', '<fin>']
logits = np.array([2.0, 1.2, 0.1, -0.4])
probabilidades = softmax(logits)

pd.DataFrame({
    'token': candidatos,
    'logit': logits,
    'probabilidad': probabilidades,
})

## Pérdida de entropía cruzada

Si el token correcto era `baja`, interesa que su probabilidad sea alta. Una pérdida común es el negativo del logaritmo de la probabilidad correcta:

$$
\mathcal{L}=-\log P(\text{token correcto})
$$

- si el modelo asigna probabilidad cercana a 1 al token correcto, la pérdida es pequeña;
- si asigna probabilidad cercana a 0, la pérdida es grande.

In [ ]:
indice_correcto = candidatos.index('baja')
perdida = -np.log(probabilidades[indice_correcto])
print(f'Probabilidad del token correcto: {probabilidades[indice_correcto]:.3f}')
print(f'Pérdida: {perdida:.3f}')

## Ejemplo desarrollado · entrenar una red mínima

La siguiente red aprende pares “token actual → token siguiente”. Tiene:

- una matriz de embeddings `E`;
- una matriz de salida `W` y un sesgo `b`;
- una pérdida de entropía cruzada;
- descenso de gradiente.

**No es un transformer**: todavía mira un solo token. Su función pedagógica es mostrar que “aprender” significa ajustar parámetros para reducir una pérdida.

In [ ]:
# Construimos el vocabulario y todos los pares de entrenamiento.
secuencias = [
    ['<inicio>'] + tokenizar_simple(oracion) + ['<fin>']
    for oracion in corpus
]
vocabulario = sorted({token for secuencia in secuencias for token in secuencia})
tok2id = {token: i for i, token in enumerate(vocabulario)}
id2tok = {i: token for token, i in tok2id.items()}

pares = [
    (tok2id[actual], tok2id[siguiente])
    for secuencia in secuencias
    for actual, siguiente in zip(secuencia[:-1], secuencia[1:])
]
X_ids = np.array([p[0] for p in pares])
y_ids = np.array([p[1] for p in pares])

print('Tamaño del vocabulario:', len(vocabulario))
print('Pares de entrenamiento:', len(pares))
print('Primeros cinco pares:', [(id2tok[x], id2tok[y]) for x, y in pares[:5]])

In [ ]:
dimension_embedding = 8
n_vocab = len(vocabulario)
rng_entrenamiento = np.random.default_rng(SEMILLA)

E = rng_entrenamiento.normal(0, 0.15, size=(n_vocab, dimension_embedding))
W = rng_entrenamiento.normal(0, 0.15, size=(dimension_embedding, n_vocab))
b = np.zeros(n_vocab)

tasa_aprendizaje = 0.35
epocas = 500
historial_perdida = []

for epoca in range(epocas):
    # Paso hacia adelante
    H = E[X_ids]                 # búsqueda de embeddings
    logits_lote = H @ W + b      # puntaje para cada token posible
    probs_lote = softmax(logits_lote, axis=1)
    perdida_media = -np.log(probs_lote[np.arange(len(y_ids)), y_ids] + 1e-12).mean()
    historial_perdida.append(perdida_media)

    # Gradientes de la entropía cruzada con softmax
    grad_logits = probs_lote.copy()
    grad_logits[np.arange(len(y_ids)), y_ids] -= 1
    grad_logits /= len(y_ids)

    grad_W = H.T @ grad_logits
    grad_b = grad_logits.sum(axis=0)
    grad_H = grad_logits @ W.T
    grad_E = np.zeros_like(E)
    np.add.at(grad_E, X_ids, grad_H)

    # Actualización de parámetros
    E -= tasa_aprendizaje * grad_E
    W -= tasa_aprendizaje * grad_W
    b -= tasa_aprendizaje * grad_b

print(f'Pérdida inicial: {historial_perdida[0]:.3f}')
print(f'Pérdida final:   {historial_perdida[-1]:.3f}')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(historial_perdida, color='#7a1731')
ax.set(title='La red ajusta parámetros para reducir la pérdida',
       xlabel='Época', ylabel='Entropía cruzada media')
plt.show()

In [ ]:
def prediccion_red_minima(token_actual, n=5):
    if token_actual not in tok2id:
        return pd.DataFrame(columns=['siguiente', 'probabilidad'])
    h = E[tok2id[token_actual]]
    probs = softmax(h @ W + b)
    indices = np.argsort(probs)[::-1][:n]
    return pd.DataFrame({
        'siguiente': [id2tok[i] for i in indices],
        'probabilidad': probs[indices],
    })


prediccion_red_minima('inflación')

### Discusión guiada

1. ¿Por qué la pérdida disminuye?
2. ¿La red aprendió una ley económica o regularidades del corpus?
3. Si el corpus contuviera un error repetido muchas veces, ¿qué esperaría que ocurriera?

<details>
<summary>Contraste conceptual</summary>

1. Porque los parámetros se actualizaron en la dirección que aumenta la probabilidad de los tokens observados como correctos.
2. Aprendió regularidades del corpus diminuto; no identificó una ley causal.
3. El error también puede convertirse en un patrón aprendido. Volumen y repetición no garantizan calidad ni verdad.
</details>

# 5. Embeddings: coordenadas aprendidas

Un **embedding** es un vector numérico que representa un token. En vez de describir `inflación` con una definición escrita, el modelo usa algo como:

$$
\text{inflación}\rightarrow [0.17,-0.83,0.42,\ldots]
$$

Las dimensiones no suelen tener una etiqueta humana directa. Durante el entrenamiento, tokens que funcionan en contextos parecidos pueden terminar en regiones cercanas del espacio.

Dos aclaraciones:

- cercanía geométrica no equivale automáticamente a sinonimia;
- el significado que usa el transformer se vuelve **contextual** después de varias capas. `banco` en “crédito bancario” y en “banco del parque” puede terminar representado de manera diferente.

## Ejemplo intuitivo · un mapa inventado de dos dimensiones

Para poder dibujarlo, usaremos solo dos dimensiones con nombres interpretables. Estos vectores son **diseñados para enseñar**, no aprendidos por un LLM real.

In [ ]:
embeddings_didacticos = pd.DataFrame(
    {
        'presion_precios': [1.0, 0.9, -0.8, -0.7, 0.1, -0.1],
        'actividad_laboral': [0.0, 0.1, 0.1, 0.0, 1.0, -1.0],
    },
    index=['inflación', 'precios', 'deflación', 'desinflación', 'empleo', 'desempleo'],
)

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(embeddings_didacticos['presion_precios'],
           embeddings_didacticos['actividad_laboral'], s=70, color='#7a1731')
for palabra, fila in embeddings_didacticos.iterrows():
    ax.annotate(palabra, (fila['presion_precios'], fila['actividad_laboral']),
                xytext=(6, 5), textcoords='offset points')
ax.axhline(0, color='gray', linewidth=.7)
ax.axvline(0, color='gray', linewidth=.7)
ax.set(xlabel='Dimensión didáctica: presión de precios',
       ylabel='Dimensión didáctica: actividad laboral',
       title='Un embedding es una posición en un espacio numérico')
plt.show()

In [ ]:
def similitud_coseno(vector_a, vector_b):
    numerador = np.dot(vector_a, vector_b)
    denominador = np.linalg.norm(vector_a) * np.linalg.norm(vector_b)
    return numerador / denominador


comparaciones = [
    ('inflación', 'precios'),
    ('empleo', 'desempleo'),
    ('inflación', 'empleo'),
]

pd.DataFrame([
    {
        'palabra_a': a,
        'palabra_b': b,
        'similitud_coseno': similitud_coseno(
            embeddings_didacticos.loc[a].to_numpy(),
            embeddings_didacticos.loc[b].to_numpy(),
        ),
    }
    for a, b in comparaciones
])

### Su turno 3 · interpretar, no memorizar

Agregue una fila para `salarios` o `recesión`, eligiendo sus dos coordenadas. Luego explique:

1. por qué eligió esa posición;
2. con qué palabra quedó más cerca;
3. por qué esta cercanía no basta para afirmar una relación causal.

In [ ]:
embeddings_estudiante = embeddings_didacticos.copy()

# Ejemplo de estructura. Cambie el término y las coordenadas.
embeddings_estudiante.loc['salarios'] = [0.4, 0.6]

nueva_palabra = 'salarios'
similitudes = {
    palabra: similitud_coseno(
        embeddings_estudiante.loc[nueva_palabra].to_numpy(),
        embeddings_estudiante.loc[palabra].to_numpy(),
    )
    for palabra in embeddings_didacticos.index
}

print(sorted(similitudes.items(), key=lambda par: par[1], reverse=True))
respuesta_embedding = 'Escriba aquí su justificación.'
print(respuesta_embedding)

# 6. Atención: decidir qué partes del contexto importan

Una limitación del bigrama es que solo mira el token anterior. Para usar un contexto amplio, el transformer incorpora **autoatención** (*self-attention*).

Intuición: cada token formula una consulta y compara esa consulta con los demás tokens disponibles.

| Nombre | Sigla | Pregunta intuitiva |
|---|---|---|
| **Query** | Q | ¿Qué información estoy buscando? |
| **Key** | K | ¿Qué tipo de información ofrezco? |
| **Value** | V | ¿Qué contenido aportaré si resulto relevante? |

Q, K y V son vectores producidos mediante matrices aprendidas. No son preguntas escritas en lenguaje natural.

## Atención de producto punto escalado

$$
\operatorname{Atención}(Q,K,V)
=\operatorname{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}+M\right)V
$$

Paso a paso:

1. $QK^\top$ mide compatibilidad entre consultas y claves;
2. $\sqrt{d_k}$ evita puntajes demasiado grandes;
3. $M$ es una máscara opcional;
4. `softmax` convierte puntajes en pesos que suman uno;
5. los valores `V` se combinan usando esos pesos.

## Ejemplo desarrollado · una cabeza de atención causal

Usaremos la secuencia `el banco subió la tasa`. Para mantener visibles los cálculos:

- cada token tendrá un vector de tres dimensiones;
- las matrices Q, K y V serán pequeñas;
- aplicaremos una **máscara causal**: cada posición solo puede mirar su posición y las anteriores.

Un modelo generador necesita esta máscara durante el entrenamiento para no “ver el futuro” que debe predecir.

In [ ]:
tokens_atencion = ['el', 'banco', 'subió', 'la', 'tasa']

X_atencion = np.array([
    [0.2, 0.0, 0.1],   # el
    [0.1, 0.9, 0.3],   # banco
    [0.2, 0.5, 1.0],   # subió
    [0.2, 0.0, 0.1],   # la
    [0.1, 1.0, 0.8],   # tasa
])

W_Q = np.array([[1.0, 0.0, 0.2], [0.0, 0.8, 0.1], [0.1, 0.0, 1.0]])
W_K = np.array([[0.9, 0.1, 0.0], [0.0, 1.0, 0.2], [0.2, 0.0, 0.8]])
W_V = np.eye(3)

Q = X_atencion @ W_Q
K = X_atencion @ W_K
V = X_atencion @ W_V

puntajes = Q @ K.T / np.sqrt(K.shape[1])
mascara_futuro = np.triu(np.ones_like(puntajes, dtype=bool), k=1)
puntajes_enmascarados = np.where(mascara_futuro, -1e9, puntajes)
pesos_atencion = softmax(puntajes_enmascarados, axis=1)
salida_atencion = pesos_atencion @ V

pd.DataFrame(pesos_atencion, index=tokens_atencion, columns=tokens_atencion).round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    pesos_atencion,
    annot=True,
    fmt='.2f',
    cmap='rocket_r',
    xticklabels=tokens_atencion,
    yticklabels=tokens_atencion,
    vmin=0,
    vmax=1,
    ax=ax,
)
ax.set(xlabel='Token del que se toma información (K/V)',
       ylabel='Token que consulta (Q)',
       title='Pesos de una cabeza de atención causal didáctica')
plt.show()

### Cómo leer el mapa

- Cada **fila** corresponde al token que está actualizando su representación.
- Cada **columna** corresponde a una posición de la que podría tomar información.
- Cada fila suma aproximadamente uno.
- El triángulo superior vale cero por la máscara causal.
- Un peso alto no demuestra una explicación causal del modelo; indica una combinación interna en esta cabeza y capa.

En un transformer real hay varias **cabezas de atención** en paralelo. Cada una puede aprender patrones diferentes; después sus salidas se combinan.

### Su turno 4 · leer la atención

Use la tabla `pesos_atencion` para responder:

1. ¿A qué posiciones puede mirar `subió`? ¿A cuáles no?
2. ¿Qué token anterior recibe mayor peso cuando se actualiza `tasa`?
3. ¿Por qué todas las celdas a la derecha de la diagonal son cero?
4. ¿Por qué sería incorrecto afirmar que “el mapa revela toda la razón del modelo”?

In [ ]:
tabla_atencion = pd.DataFrame(
    pesos_atencion,
    index=tokens_atencion,
    columns=tokens_atencion,
)

fila_tasa = tabla_atencion.loc['tasa']
print('Pesos usados al actualizar “tasa”:')
display(fila_tasa.sort_values(ascending=False).to_frame('peso'))

respuesta_atencion = 'Escriba aquí sus respuestas 1–4.'
print(respuesta_atencion)

# 7. ¿Qué es un transformer?

Un **transformer** es una arquitectura de red neuronal diseñada para procesar secuencias mediante atención. Un bloque típico contiene:

| Componente | Función |
|---|---|
| Embedding de token | Convierte identificadores en vectores. |
| Información posicional | Permite distinguir orden y posición. |
| Atención multi-cabeza | Combina información de distintas posiciones y patrones. |
| Red *feed-forward* | Transforma cada posición con una red neuronal pequeña compartida. |
| Conexiones residuales | Conservan y suman información a través de capas. |
| Normalización | Ayuda a estabilizar el entrenamiento. |

Un LLM generativo suele apilar muchos bloques. Al final, una capa produce logits sobre el vocabulario para el siguiente token.

> **Transformer** nombra la arquitectura. **LLM** describe un modelo de lenguaje grande; muchos LLM actuales usan transformers, pero los dos términos no son sinónimos perfectos.

In [ ]:
from IPython.display import HTML

# Código iframe estándar de YouTube
HTML("""
<iframe width="560" height="315" 
src="https://www.youtube.com/embed/LPZh9BOjkQs?si=SxuvQ9-aarRhtB1T" 
title="YouTube video player" 
frameborder="0" 
allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture; web-share" 
referrerpolicy="strict-origin-when-cross-origin" 
allowfullscreen>
</iframe>
""")



Por si no funciona, el link del video es: https://www.youtube.com/watch?v=LPZh9BOjkQs

## ¿Por qué hace falta información de posición?

La atención, por sí sola, compara vectores sin conocer automáticamente el orden. Por eso se agrega una representación posicional.

Compare:

- “el banco subió la tasa”;
- “la tasa subió el banco”.

Contienen casi los mismos tokens, pero el orden cambia el sentido. El ejemplo siguiente dibuja una codificación sinusoidal, una de las formas históricas de incorporar posición.

In [ ]:
def codificacion_posicional(n_posiciones, dimension):
    posiciones = np.arange(n_posiciones)[:, None]
    indices = np.arange(dimension)[None, :]
    tasas = 1 / np.power(10000, (2 * (indices // 2)) / dimension)
    angulos = posiciones * tasas
    codificacion = np.zeros((n_posiciones, dimension))
    codificacion[:, 0::2] = np.sin(angulos[:, 0::2])
    codificacion[:, 1::2] = np.cos(angulos[:, 1::2])
    return codificacion


posiciones = codificacion_posicional(n_posiciones=8, dimension=10)

fig, ax = plt.subplots(figsize=(9, 4))
sns.heatmap(posiciones, cmap='coolwarm', center=0, ax=ax)
ax.set(xlabel='Dimensión', ylabel='Posición en la secuencia',
       title='Cada posición recibe un patrón numérico diferente')
plt.show()

## Del bloque al modelo grande

La conexión completa puede resumirse así:

1. **Texto:** “La inflación anual…”
2. **Tokens e identificadores:** `[La] [inflación] [anual] ...`
3. **Embeddings + posición:** vectores numéricos con identidad y orden.
4. **Bloques transformer:** atención + red neuronal + normalización + residuales.
5. **Logits:** un puntaje por token del vocabulario.
6. **Probabilidades:** `softmax`.
7. **Selección:** se elige o muestrea un token.
8. **Repetición:** el nuevo token se agrega al contexto y el ciclo continúa.

La salida completa aparece token por token, aunque la interfaz la muestre como un párrafo fluido.

### Comprobación rápida 2

Complete mentalmente:

1. El embedding convierte un token en __________.
2. La atención permite __________ información de diferentes posiciones.
3. La máscara causal impide mirar __________.
4. El transformer completo sigue siendo una __________ neuronal entrenada al ajustar parámetros.

<details>
<summary>Ver respuestas</summary>

1. un vector numérico;
2. ponderar o combinar;
3. tokens futuros;
4. red.
</details>

# 8. Temperatura y generación

La **temperatura** transforma los logits antes de `softmax`:

$$
P_i(T)=\operatorname{softmax}\left(\frac{z_i}{T}\right)
$$

- temperatura baja: concentra probabilidad en las opciones dominantes;
- temperatura alta: aplana la distribución y aumenta diversidad;
- temperatura alta **no añade conocimiento** ni vuelve correcta una respuesta.

No todos los modelos o modos de API exponen exactamente los mismos controles. Consulte la documentación del modelo que use.

In [ ]:
tokens_temperatura = ['aumentó', 'disminuyó', 'permaneció', 'sorprendió']
logits_temperatura = np.array([2.4, 1.5, 0.8, -0.2])

filas = []
for temperatura in [0.3, 0.7, 1.0, 1.8]:
    probs = softmax(logits_temperatura / temperatura)
    for token, prob in zip(tokens_temperatura, probs):
        filas.append({'temperatura': temperatura, 'token': token, 'probabilidad': prob})

tabla_temperatura = pd.DataFrame(filas)
tabla_temperatura.pivot(index='token', columns='temperatura', values='probabilidad').round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.lineplot(
    data=tabla_temperatura,
    x='temperatura',
    y='probabilidad',
    hue='token',
    marker='o',
    ax=ax,
)
ax.set(title='La temperatura cambia la distribución, no el conocimiento',
       ylim=(0, 1))
plt.show()

### Su turno 5 · hacer una predicción antes de ejecutar

Cambie `temperatura_estudiante` por `0.2` y luego por `2.0`.

Antes de ejecutar, prediga:

- ¿qué ocurrirá con la probabilidad de `aumentó`?;
- ¿qué ocurrirá con las opciones menos probables?;
- ¿cuál configuración esperaría que produzca más variedad si se muestrea varias veces?

In [ ]:
temperatura_estudiante = 0.2
probs_estudiante = softmax(logits_temperatura / temperatura_estudiante)

pd.DataFrame({
    'token': tokens_temperatura,
    'probabilidad': probs_estudiante,
}).sort_values('probabilidad', ascending=False)

# 9. ¿Cómo se entrena un LLM útil?

Aunque los procesos varían entre familias de modelos, conviene distinguir tres etapas conceptuales:

| Etapa | Objetivo simplificado | Qué aporta |
|---|---|---|
| **Preentrenamiento** | Predecir tokens en grandes colecciones de datos. | Patrones de lenguaje y capacidades generales. |
| **Ajuste con instrucciones** | Aprender a responder tareas expresadas como instrucciones. | Seguimiento de solicitudes y formatos. |
| **Posentrenamiento/alineación** | Mejorar utilidad, seguridad y preferencias de respuesta con señales humanas o sintéticas. | Conducta más útil y controlada. |

Durante el uso normal hacemos **inferencia**: aplicamos parámetros ya entrenados a una entrada. En este contexto, “inferencia” computacional no significa lo mismo que inferencia causal o estadística.

## Capacidades emergentes y límites estructurales

La escala y el entrenamiento permiten comportamientos que no programamos regla por regla. Sin embargo:

| Puede ayudar a… | No garantiza… |
|---|---|
| resumir y reorganizar texto | fidelidad perfecta a la fuente |
| proponer código | código correcto, seguro o eficiente |
| clasificar documentos | ausencia de sesgo o error |
| redactar explicaciones | comprensión humana o conciencia |
| generar hipótesis | evidencia causal |
| responder con fluidez | verdad, actualidad o trazabilidad |

Una **alucinación** es una salida presentada con aparente seguridad que no está respaldada por los datos, las fuentes o los hechos. Es coherente con el mecanismo: el modelo optimiza una continuación, no una prueba automática de verdad.

## Caso económico breve

Suponga que un LLM responde:

> “La reforma redujo el desempleo en 2,3 puntos porcentuales durante 2025.”

Antes de usar esta frase en un informe, harían falta por lo menos:

1. identificar la fuente exacta y la definición de desempleo;
2. comprobar el periodo y la población;
3. distinguir cambio observado de efecto causal;
4. verificar el cálculo contra datos originales;
5. documentar el modelo, la entrada y la fecha de consulta.

**Pregunta:** ¿cuáles de esos cinco pasos puede proponer un LLM y cuáles siguen siendo responsabilidad del analista?

In [ ]:
respuesta_caso = (
    'Escriba aquí una respuesta de 4–6 líneas. '
    'Distinga ayuda operativa, verificación y juicio profesional.'
)
print(respuesta_caso)

# 10. ¿Qué significa llamar un LLM desde Python?

Una **API** es un contrato para que dos programas se comuniquen. En este caso:

1. Python construye una solicitud;
2. el SDK la envía al servicio;
3. el servicio ejecuta el modelo;
4. devuelve un objeto de respuesta;
5. Python extrae y procesa el texto.

El SDK oficial evita que tengamos que construir manualmente cada solicitud HTTP. Aun así, el analista debe definir instrucciones, entrada, modelo, controles, validación y registro.

## 10.1 Preparar el SDK y la clave

Instalación del paquete oficial:

```python
%pip install -U openai
```

La clave debe guardarse como variable de entorno llamada `OPENAI_API_KEY`. **Nunca** la escriba directamente en una celda, un repositorio, un correo o un archivo compartido.

En macOS o Linux, desde una terminal:

```bash
export OPENAI_API_KEY="su_clave_aquí"
```

En PowerShell:

```powershell
setx OPENAI_API_KEY "su_clave_aquí"
```

Después abra un kernel nuevo. El SDK lee la variable de entorno automáticamente.

> Tener ChatGPT no implica necesariamente tener créditos de API. El uso programático puede tener facturación y límites separados.

In [ ]:
hay_clave = bool(os.getenv('OPENAI_API_KEY'))
print('¿Hay una variable OPENAI_API_KEY disponible?', hay_clave)
print('La clave no se imprime ni se guarda en este cuaderno.')

## 10.2 Separar instrucciones y entrada

- `instructions` define comportamiento general, criterios y formato.
- `input` contiene la tarea o material concreto del usuario.

Separarlos hace el programa más legible. Ningún prompt elimina por sí solo el riesgo de error; la salida debe validarse.

In [ ]:
INSTRUCCIONES = """
Actúa como asistente de análisis económico para estudiantes universitarios.
Explica en español claro y usa máximo 120 palabras.
No inventes cifras ni fuentes. Si falta evidencia, indícalo explícitamente.
Distingue descripción, predicción y afirmación causal.
""".strip()

ENTRADA = """
Una ciudad observó que el desempleo bajó después de implementar un programa de capacitación.
Explica por qué esa comparación antes-después no basta para afirmar que el programa causó la reducción.
Propón dos datos o diseños que fortalecerían el análisis.
""".strip()

print('INSTRUCCIONES:\n', INSTRUCCIONES)
print('\nENTRADA:\n', ENTRADA)

## 10.3 Primera llamada con la Responses API

La celda usa el patrón actual del SDK oficial:

```python
from openai import OpenAI

client = OpenAI()
response = client.responses.create(
    model="gpt-6-astra",
    instructions=INSTRUCCIONES,
    input=ENTRADA,
)
print(response.output_text)
```

`response.output_text` reúne el texto de salida de forma conveniente. No conviene suponer que el texto siempre está en una posición fija de la estructura interna de `response.output`.

El nombre del modelo puede depender de la cuenta, disponibilidad y fecha. Por eso el código permite reemplazarlo mediante `OPENAI_MODEL`.

In [ ]:
MODELO = os.getenv('OPENAI_MODEL', 'gpt-6-astra')

# Seguridad pedagógica: cambie a True solo con autorización del docente.
EJECUTAR_LLAMADA_REAL = False

RESPUESTA_LLM = None
ID_RESPUESTA = None
USO_RESPUESTA = None

if not EJECUTAR_LLAMADA_REAL:
    print('Llamada omitida: EJECUTAR_LLAMADA_REAL está en False.')
elif not os.getenv('OPENAI_API_KEY'):
    print('Llamada omitida: no existe OPENAI_API_KEY en el entorno.')
else:
    try:
        from openai import OpenAI
    except ImportError:
        print('Instale primero el SDK con: %pip install -U openai')
    else:
        client = OpenAI()
        response = client.responses.create(
            model=MODELO,
            instructions=INSTRUCCIONES,
            input=ENTRADA,
        )
        RESPUESTA_LLM = response.output_text
        ID_RESPUESTA = getattr(response, 'id', None)
        USO_RESPUESTA = getattr(response, 'usage', None)
        print(RESPUESTA_LLM)

## 10.4 Encapsular la llamada en una función

Una función reduce copia y pega, permite validar argumentos y facilita pruebas. La función siguiente:

- exige una clave disponible;
- recibe entrada, instrucciones y modelo;
- devuelve texto y metadatos mínimos;
- no guarda la clave.

In [ ]:
def consultar_llm(entrada, instrucciones, modelo=None):
    """Envía una solicitud de texto y devuelve salida más metadatos básicos."""
    if not isinstance(entrada, str) or not entrada.strip():
        raise ValueError('La entrada debe ser un texto no vacío.')
    if not isinstance(instrucciones, str) or not instrucciones.strip():
        raise ValueError('Las instrucciones deben ser un texto no vacío.')
    if not os.getenv('OPENAI_API_KEY'):
        raise RuntimeError('No se encontró OPENAI_API_KEY en el entorno.')

    try:
        from openai import OpenAI
    except ImportError as exc:
        raise ImportError('Instale el SDK con: %pip install -U openai') from exc

    modelo_elegido = modelo or os.getenv('OPENAI_MODEL', 'gpt-6-astra')
    cliente = OpenAI()
    respuesta = cliente.responses.create(
        model=modelo_elegido,
        instructions=instrucciones,
        input=entrada,
    )
    return {
        'texto': respuesta.output_text,
        'response_id': getattr(respuesta, 'id', None),
        'modelo_solicitado': modelo_elegido,
        'uso': getattr(respuesta, 'usage', None),
    }


print('Función definida. No se ha enviado ninguna solicitud.')

## 10.5 Adaptar la solicitud

Observe que no necesitamos cambiar la función. Solo cambiamos los datos:

- **objetivo:** clasificar, resumir, explicar o extraer;
- **contexto:** texto que el modelo debe analizar;
- **restricciones:** extensión, tono, fuentes permitidas;
- **formato de salida:** lista, tabla textual, JSON, etc.;
- **criterio de incertidumbre:** qué hacer cuando falta información.

In [ ]:
INSTRUCCIONES_ESTUDIANTE = """
Actúa como revisor metodológico. Responde en español con tres apartados:
1. Qué sí permiten afirmar los datos.
2. Qué no permiten afirmar.
3. Qué información falta.
No inventes resultados.
""".strip()

ENTRADA_ESTUDIANTE = """
En una encuesta transversal de 300 hogares, quienes usan banca móvil reportan mayor ahorro promedio.
Interpreta el hallazgo sin convertir una asociación en causalidad.
""".strip()

if EJECUTAR_LLAMADA_REAL:
    resultado_estudiante = consultar_llm(
        entrada=ENTRADA_ESTUDIANTE,
        instrucciones=INSTRUCCIONES_ESTUDIANTE,
        modelo=MODELO,
    )
    print(resultado_estudiante['texto'])
else:
    resultado_estudiante = None
    print('Ejemplo preparado. Active la llamada solo cuando el docente lo indique.')

### Su turno 6 · diseñar y evaluar una llamada

En pareja, adapten `INSTRUCCIONES_ESTUDIANTE` y `ENTRADA_ESTUDIANTE` para una de estas tareas:

1. resumir un párrafo económico sin añadir cifras;
2. clasificar una noticia como descripción, predicción o afirmación causal;
3. explicar un fragmento de código de `pandas` línea por línea.

Antes de ejecutar, escriban:

- qué resultado esperan;
- dos criterios con los que evaluarán la respuesta;
- un error plausible que deberán revisar.

Después de ejecutar, no respondan solo “funcionó”. Comparen la salida con sus criterios.

In [ ]:
TAREA_ELEGIDA = 'Escriba aquí 1, 2 o 3.'
INSTRUCCIONES_PAREJA = 'Escriban aquí instrucciones completas.'
ENTRADA_PAREJA = 'Peguen aquí el texto o código que analizarán.'

CRITERIOS_EVALUACION = [
    'Criterio 1:',
    'Criterio 2:',
]
ERROR_PLAUSIBLE = 'Error que revisaremos:'

print('Tarea:', TAREA_ELEGIDA)
print('Criterios:', CRITERIOS_EVALUACION)
print(ERROR_PLAUSIBLE)

In [ ]:
# Esta celda tampoco ejecuta nada mientras la bandera sea False.
if EJECUTAR_LLAMADA_REAL:
    resultado_pareja = consultar_llm(
        entrada=ENTRADA_PAREJA,
        instrucciones=INSTRUCCIONES_PAREJA,
        modelo=MODELO,
    )
    print(resultado_pareja['texto'])
else:
    resultado_pareja = None
    print('Llamada de la pareja omitida de forma segura.')

## 10.6 Trazabilidad mínima

Una salida útil debe poder reconstruirse. Registre al menos:

- fecha y hora;
- modelo solicitado;
- instrucciones;
- entrada;
- identificador de respuesta, si existe;
- salida;
- resultado de la revisión humana.

No incluya claves ni datos sensibles. Si la entrada contiene información personal o confidencial, no la envíe sin autorización y reglas institucionales claras.

In [ ]:
registro_llamada = pd.DataFrame([{
    'fecha_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'),
    'modelo_solicitado': MODELO,
    'instructions': INSTRUCCIONES,
    'input': ENTRADA,
    'response_id': ID_RESPUESTA,
    'output': RESPUESTA_LLM if RESPUESTA_LLM is not None else 'No ejecutada',
    'revision_humana': 'Pendiente' if RESPUESTA_LLM else 'No aplica',
}])

registro_llamada.T

# 11. Actividad de cierre · explicar el recorrido completo

Sin consultar las respuestas anteriores, complete con su pareja:

1. Un LLM recibe texto, lo divide en __________ y los convierte en identificadores.
2. Un __________ transforma cada identificador en un vector.
3. La __________ combina información del contexto usando pesos.
4. Un transformer repite bloques de atención y redes __________.
5. La capa final produce __________; `softmax` los convierte en probabilidades.
6. El modelo genera seleccionando el siguiente token y __________ el proceso.
7. Una salida probable no es necesariamente __________.
8. Desde Python usamos una __________ para comunicarnos con el servicio.
9. La clave debe guardarse en una variable de __________.
10. La salida debe pasar por __________ humana y quedar registrada.

In [ ]:
respuestas_cierre = {
    1: '',
    2: '',
    3: '',
    4: '',
    5: '',
    6: '',
    7: '',
    8: '',
    9: '',
    10: '',
}

pd.Series(respuestas_cierre, name='respuesta')

<details>
<summary>Ver respuestas de cierre</summary>

1. tokens;
2. embedding;
3. atención;
4. neuronales *feed-forward*;
5. logits;
6. repitiendo;
7. verdadera;
8. API;
9. entorno;
10. revisión o validación.
</details>

# 12. Lista de verificación

Antes de cerrar, compruebe si puede explicar con sus propias palabras:

- [ ] la diferencia entre palabra y token;
- [ ] la predicción del siguiente token;
- [ ] parámetros, logits, `softmax` y pérdida;
- [ ] embeddings estáticos y representaciones contextuales;
- [ ] Q, K y V sin tratarlos como texto literal;
- [ ] la máscara causal;
- [ ] transformer frente a LLM;
- [ ] temperatura frente a conocimiento;
- [ ] por qué ocurre una alucinación;
- [ ] cómo llamar un modelo desde Python sin exponer la clave;
- [ ] por qué hay que registrar y validar la salida.

# Glosario mínimo

| Término | Definición breve |
|---|---|
| **LLM** | *Large Language Model*: modelo grande de lenguaje. |
| **Token** | Unidad que procesa el modelo: palabra, subpalabra, signo u otra unidad. |
| **Parámetro** | Número aprendido que modifica las transformaciones de la red. |
| **Embedding** | Vector numérico asociado a un token o fragmento. |
| **Logit** | Puntaje previo a convertirlo en probabilidad. |
| **Softmax** | Función que transforma logits en una distribución de probabilidad. |
| **Atención** | Mecanismo que pondera y combina información de varias posiciones. |
| **Transformer** | Arquitectura neuronal basada en atención y bloques repetidos. |
| **Prompt** | Entrada e instrucciones que orientan la generación. |
| **API** | Interfaz mediante la cual un programa solicita un servicio a otro. |
| **SDK** | Paquete de herramientas que simplifica el uso de una API. |
| **Inferencia computacional** | Ejecución de un modelo ya entrenado sobre una entrada. |
| **Alucinación** | Salida no respaldada presentada como si fuera válida. |
| **Trazabilidad** | Capacidad de reconstruir entrada, configuración, salida y revisión. |

# Referencias y documentación

- Vaswani, A. et al. (2017). *Attention Is All You Need*. https://arxiv.org/abs/1706.03762
- Jurafsky, D. y Martin, J. H. *Speech and Language Processing*. https://web.stanford.edu/~jurafsky/slp3/
- OpenAI. *Developer quickstart*. https://developers.openai.com/api/docs/quickstart
- OpenAI. *Text generation*. https://developers.openai.com/api/docs/guides/text

La sintaxis de API puede cambiar. Para usos reales, confirme siempre el nombre del modelo, los parámetros admitidos, los precios y las políticas vigentes en la documentación oficial.